<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/CSC791_DLBA_ResNet_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split, Subset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## ImageNet-style Transforms

In [3]:
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=16),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

## Load the CIFAR-10 DATASET

#### Using torchvision

In [4]:
import torch
import torchvision
from torch.utils.data import DataLoader, random_split, Subset

# Get transforms
train_transform, eval_transform = get_transforms(img_size=224)

# Load CIFAR-10 training set twice:
# once with train augmentation, once with eval transform
full_train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

full_train_dataset_eval = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=eval_transform
)

# Official test set
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=eval_transform
)

# Split original training set into train and validation
total_size = len(full_train_dataset)
val_size = int(0.1 * total_size)   # 10% validation
train_size = total_size - val_size

generator = torch.Generator().manual_seed(42)

train_subset_tmp, val_subset_tmp = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)

train_indices = train_subset_tmp.indices
val_indices = val_subset_tmp.indices

trainset = Subset(full_train_dataset, train_indices)
valset = Subset(full_train_dataset_eval, val_indices)

# DataLoaders
trainloader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

valloader = DataLoader(
    valset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print(f"Train samples: {len(trainset)}")
print(f"Validation samples: {len(valset)}")
print(f"Test samples: {len(testset)}")
print("CIFAR-10 dataset loaded successfully.")

100%|██████████| 170M/170M [00:03<00:00, 48.8MB/s]


Train samples: 45000
Validation samples: 5000
Test samples: 10000
CIFAR-10 dataset loaded successfully.


## Get the RESNET18 Model

---



### Load Resnet18 with Pre-trained Weights on the ImageNet 1K dataset

In [5]:
# Load the ResNet-18 model with default ImageNet1K V1 weights
# resnet18_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Stage Setup Functions for Fine-Tuning

In [6]:
def replace_resnet_classifier(model, num_classes=10):
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def freeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True


def set_finetune_stage(model, stage):
    """
    stage 1: train only final classifier head
    stage 2: train layer4 + classifier head
    stage 3: train full model
    """
    freeze_all_layers(model)

    if stage == 1:
        unfreeze_module(model.fc)

    elif stage == 2:
        unfreeze_module(model.layer4)
        unfreeze_module(model.fc)

    elif stage == 3:
        unfreeze_module(model)

    else:
        raise ValueError("stage must be 1, 2, or 3")

    return model

# Training and Evaluation Functions

Defines reusable functions to train the model for one epoch and evaluate it on validation/test data.

In [7]:
import torch

def train_one_epoch(model, trainloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

# Run One Fine-Tuning Stage

Runs training + evaluation for a given stage (e.g., head-only, partial, full fine-tuning) and tracks metrics.

In [8]:
def run_stage(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    stage,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    best_model_path="best_model_cifar10_resnet18.pth"
):
    history = {
        "stage": stage,
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    print(f"\nStarting Stage {stage} for {epochs} epoch(s)\n")

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Stage {stage} | "
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Optimizer + Debug Utilities

Defines optimizer setup (with learning rates) and a helper to verify which layers are trainable.

In [9]:
import torch.optim as optim

def get_stage_optimizer_sgd(model, stage):
    momentum = 0.9
    weight_decay = 1e-4

    if stage == 1:
        lr = 1e-2
        print(f"Stage 1 LR (fc): {lr}")
        return optim.SGD(
            model.fc.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 2:
        lr_backbone = 1e-3
        lr_head = 5e-3
        print(f"Stage 2 LR (layer4): {lr_backbone}, LR (fc): {lr_head}")
        return optim.SGD(
            [
                {"params": model.layer4.parameters(), "lr": lr_backbone},
                {"params": model.fc.parameters(), "lr": lr_head},
            ],
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 3:
        lr = 1e-4
        print(f"Stage 3 LR (full model): {lr}")
        return optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("stage must be 1, 2, or 3")

def get_stage_optimizer_adam(model, stage, optimizer_name="adam"):
    """
    Returns optimizer with stage-specific learning rates
    """

    if optimizer_name.lower() == "adam":

        if stage == 1:
            # only classifier
            lr = 1e-3
            print(f"Stage 1 LR (fc): {lr}")
            return optim.Adam(model.fc.parameters(), lr=lr)

        elif stage == 2:
            # layer4 + classifier
            lr_backbone = 1e-4
            lr_head = 1e-3
            print(f"Stage 2 LR (layer4): {lr_backbone}, LR (fc): {lr_head}")
            return optim.Adam([
                {"params": model.layer4.parameters(), "lr": lr_backbone},
                {"params": model.fc.parameters(), "lr": lr_head}
            ])

        elif stage == 3:
            # full model
            lr = 1e-5
            print(f"Stage 3 LR (full model): {lr}")
            return optim.Adam(model.parameters(), lr=lr)

        else:
            raise ValueError("stage must be 1, 2, or 3")

    else:
        raise ValueError("Only 'adam' implemented for now")


def print_trainable_parameters(model):
    """
    Prints which layers are currently trainable
    """
    print("\nTrainable parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(name)

def get_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Run Full Fine-Tuning Pipeline (Stages 1 → 3)

Runs staged fine-tuning: first head-only, then partial layers, then full model. Keeps the same model and progressively unfreezes.

In [10]:
import torch.nn as nn
import torchvision.models as models

# load model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model = replace_resnet_classifier(model, num_classes=10)
model = model.to(device)

# loss
criterion = nn.CrossEntropyLoss()

all_history = []
best_val_acc = 0.0

# define how many epochs per stage
stage_plan = {
    1: 2,
    2: 4,
    3: 8
}

for stage, epochs in stage_plan.items():
    model = set_finetune_stage(model, stage)
    optimizer = get_stage_optimizer_sgd(model, stage)
    scheduler = get_scheduler(optimizer)

    print_trainable_parameters(model)

    stage_history, best_val_acc = run_stage(
        model=model,
        trainloader=trainloader,
        valloader=valloader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        stage=stage,
        epochs=epochs,
        scheduler=scheduler,
        best_val_acc=best_val_acc
    )

    all_history.append(stage_history)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


Stage 1 LR (fc): 0.01

Trainable parameters:
fc.weight
fc.bias

Starting Stage 1 for 2 epoch(s)

New best model saved with val acc: 76.88%
Stage 1 | Epoch [1/2] | Train Loss: 0.7572 | Train Acc: 74.13% | Val Loss: 0.6561 | Val Acc: 76.88%
New best model saved with val acc: 77.44%
Stage 1 | Epoch [2/2] | Train Loss: 0.6669 | Train Acc: 77.29% | Val Loss: 0.6738 | Val Acc: 77.44%
Stage 2 LR (layer4): 0.001, LR (fc): 0.005

Trainable parameters:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias

Starting Stage 2 for 4 epoch(s)

New best model saved with val acc: 89.30%
Stage 2 | Epoch [1/4] | Train Loss: 0.4426 | Train Acc: 84.92% | Val Loss: 0.3094 | Val Acc: 89.30%
New best model saved with val acc:

Test

In [11]:
model.load_state_dict(torch.load("best_model_cifar10_resnet18.pth"))
model = model.to(device)

test_loss, test_acc = evaluate(model, testloader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc: {test_acc:.2f}%")

Test Loss: 0.2011
Test Acc: 93.37%
